# First analysis to load AV_Dome data from RDSS and get Learning curves 

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
import os
from pathlib import Path

In [103]:
# This assumes that you mounted the AV_Dome prject folder as a mapped network drive "Z:". If you mounted it to a different location, please change the path below.
root_folder = Path(r"Z:\AV_Dome")

# list all available subjects:
dirs = [p for p in root_folder.iterdir() if p.is_dir()]

all_subjects = []
for i in dirs:
    subject = i.name.split("sub-")[-1]
    all_subjects.append(subject)
#

# ## OVERWRITE FOR NOW!!!
all_subjects = ["GN011"]

In [104]:
df = pd.DataFrame()
master_columns = None

for subject in all_subjects:
    sessionFolders = [p for p in (root_folder / f"sub-{subject}").iterdir() if p.is_dir()]

    for session in sessionFolders:
        print(session)
        try:
            data = pd.read_csv(session / "TrialSettings" / "TrialSettings.csv", header=None)
        except Exception as e:
            print(f"Loading error: {session} -> {e}")
            continue

        if data.empty:
            continue

        # Use the first row as the header row, then remove any later rows that repeat it.
        header_row = data.iloc[0].astype(str).tolist()
        data = data.iloc[1:].copy()
        data.columns = header_row

        repeated_header_mask = data.apply(
            lambda row: row.astype(str).tolist() == header_row,
            axis=1
        )
        data = data.loc[~repeated_header_mask].copy()

        # Clean column names by keeping only the final segment after the last dot.
        data = data.rename(columns=lambda col: col.rsplit(".", 1)[-1] if isinstance(col, str) else col)

        # Add the session identifier as a column.
        data["session_path"] = str(session)

        if master_columns is None:
            master_columns = list(data.columns)
            df = data.copy()
        else:
            new_columns = [col for col in data.columns if col not in master_columns]
            if new_columns:
                master_columns = master_columns + new_columns
                df = df.reindex(columns=master_columns)
                data = data.reindex(columns=master_columns)

            missing_columns = [col for col in master_columns if col not in data.columns]
            for col in missing_columns:
                data[col] = np.nan

            df = df.reindex(columns=master_columns)
            data = data.reindex(columns=master_columns)

            df = pd.concat([df, data], ignore_index=True)
    #
    
#

# Fix column name ("Response" is actually "trial ID", "Item1" is "Response", "Item2" is "target_side")
df = df.rename(columns={"Response": "trial_id", "Item1": "response", "Item2": "target_side"})

Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-29T11-03-23
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-29T11-08-29
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-30T14-38-56
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-01T12-55-50
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-02T15-32-43
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-02T15-38-39
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-03T13-50-31
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-03T13-54-10
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-15T13-26-48
Loading error: Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-15T13-26-48 -> [Errno 2] No such file or directory: 'Z:\\AV_Dome\\sub-GN011\\ses-Stage 0_date-2026-07-15T13-26-48\\TrialSettings\\TrialSettings.csv'
Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-15T13-28-22
Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-30T14-38-21


In [105]:
# Inspect df:
# print(df.iloc[0:100, 30:])

In [106]:
import re

# Add a column "training_stage" with integers given as "Stage #" and default to 0 if not found.
if "training_stage" not in df.columns:
    df["training_stage"] = 0


def get_column_series(frame, col_name):
    if not isinstance(col_name, str):
        return None
    try:
        result = frame[col_name]
    except Exception:
        return None

    if isinstance(result, pd.Series):
        return result
    if isinstance(result, pd.DataFrame) and result.shape[1] == 1:
        return result.iloc[:, 0]
    return None


stage_col = None
for col in df.columns:
    if isinstance(col, str) and col.lower().startswith("stage"):
        stage_col = col
        break

if stage_col is None:
    for col in df.columns:
        series = get_column_series(df, col)
        if series is None:
            continue
        if pd.api.types.is_object_dtype(series.dtype):
            sample = series.dropna().astype(str)
            if sample.str.contains(r"stage", case=False, na=False).any():
                stage_col = col
                break

if stage_col is not None:
    def extract_stage(value):
        if pd.isna(value):
            return 0
        if isinstance(value, str):
            m = re.search(r"stage\s*(\d+)", value.lower())
            if m:
                return int(m.group(1))
        return 0

    df["training_stage"] = df[stage_col].apply(extract_stage)

# Simplified session_id assignment from the end of each session_path.
if "session_path" in df.columns:
    session_paths = df["session_path"].dropna().astype(str).unique()

    def parse_session_id(path):
        try:
            token = str(path).split("/")[-1]
            match = re.search(r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2})", token)
            if match:
                digits = re.sub(r"\D", "", match.group(1))
                return int(digits)
            digits = re.sub(r"\D", "", token)
            return int(digits) if digits else -1
        except Exception:
            return -1

    ordered_sessions = sorted(session_paths, key=parse_session_id)
    session_id_map = {path: i for i, path in enumerate(ordered_sessions)}
    df["session_id"] = df["session_path"].astype(str).map(session_id_map).astype("int64")
#

df

,Seconds,trial_id,response,trialType,RewardProbability,IsRewarded,IsRepeat,RepeatCount,target_side,ITI,...,visualFlickerFrequency,visualOrientation,visualSpatialFrequency,visualTemporalFrequency,visualFlickerFrequency,audioElevation,audioFlickerFrequency,session_path,training_stage,session_id
0,607640502,1,NoResponse,0,1,True,False,0,Right,2.5576781723916895,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-...,0,0
1,607646953,2,NoResponse,2,1,True,False,0,Right,2.91447295686904,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-...,0,0
2,607654002,3,NoResponse,2,1,True,False,0,Left,3.4959962568227185,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-...,0,0
3,607661035,4,NoResponse,2,1,True,False,0,Left,3.4764310791978756,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-...,0,0
4,607667186,5,NoResponse,2,1,True,False,0,Left,2.5957006994149183,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-06-...,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1742,1999603905,161,NoResponse,2,1,True,False,0,Left,3.3442565057632776,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-...,0,9
1743,1999610088,162,NoResponse,2,1,True,False,0,Left,2.6252823682153981,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-...,0,9
1744,706939404,1,NoResponse,0,1,True,False,0,Right,2.6177636390168981,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-...,1,2
1745,706946020,2,NoResponse,2,1,True,False,0,Left,3.0765565659741672,...,8,0,0.066666666999999999,0,8,15,8,Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-...,1,2


In [116]:
[df.session_path[1742:], df.session_id[1742:], df.training_stage[1742:]]

[1742    Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-...
 1743    Z:\AV_Dome\sub-GN011\ses-Stage 0_date-2026-07-...
 1744    Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-...
 1745    Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-...
 1746    Z:\AV_Dome\sub-GN011\ses-Stage 1_date-2026-06-...
 Name: session_path, dtype: object,
 1742    9
 1743    9
 1744    2
 1745    2
 1746    2
 Name: session_id, dtype: int64,
 1742    0
 1743    0
 1744    1
 1745    1
 1746    1
 Name: training_stage, dtype: int64]